메타클래스로 여전히 처리할 수 없는게 있다. 다음 코드는 디버깅 데코레이터를 이용하여 메소드에 전달되는 인자, 반환값, 발생한 예외를 모두 출력하는 것이다.

In [ ]:
from functools import wraps

def trace_func(func):
    if hasattr(func, 'tracing'):  # 단 한번만 데코레이터를 적용한다
        return func

    @wraps(func)
    def wrapper(*args, **kwargs):
        result = None
        try:
            result = func(*args, **kwargs)
            return result
        except Exception as e:
            result = e
            raise
        finally:
            print(f'{func.__name__}({args!r}, {kwargs!r}) -> '
                  f'{result!r}')

    wrapper.tracing = True
    return wrapper

class TraceDict(dict):
    @trace_func
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @trace_func
    def __setitem__(self, *args, **kwargs):
        return super().__setitem__(*args, **kwargs)

    @trace_func
    def __getitem__(self, *args, **kwargs):
        return super().__getitem__(*args, **kwargs)


trace_dict = TraceDict([('안녕', 1)])
trace_dict['거기'] = 2
trace_dict['안녕']
try:
    trace_dict['존재하지 않음']
except KeyError:
    pass # 키 오류가 발생할 것으로 예상함

이 코드의 문제점은 모든 메서드에 `@trace_func` 데코레이터를 써야 한다는 점이다. 이는 가독성도 떨어트리고 실수도 유발할 수 있다.

문제를 해결하는 방법은 메타클래스를 사용해 클래스에 속한 모든 메소드를 자동으로 감싸는 것이다.

In [ ]:
import types

trace_types = (
    types.MethodType,
    types.FunctionType,
    types.BuiltinFunctionType,
    types.BuiltinMethodType,
    types.MethodDescriptorType,
    types.ClassMethodDescriptorType)

class TraceMeta(type):
    def __new__(meta, name, bases, class_dict):
        klass = super().__new__(meta, name, bases, class_dict)

        for key in dir(klass):
            value = getattr(klass, key)
            if isinstance(value, trace_types):
                wrapped = trace_func(value)
                setattr(klass, key, wrapped)

        return klass

class TraceDict(dict, metaclass=TraceMeta):
    pass

trace_dict = TraceDict([('안녕', 1)])
trace_dict['거기'] = 2
trace_dict['안녕']
try:
    trace_dict['존재하지 않음']
except KeyError:
    pass # 키 오류가 발생할 것으로 예상함

잘 작동하지만, 만약 상위 클래스가 메타클래스를 이미 정의한 경우엔?

In [ ]:
class OtherMeta(type):
    pass

class SimpleDict(dict, metaclass=OtherMeta):
    pass

class TraceDict(SimpleDict, metaclass=TraceMeta):
    pass

`OtherMeta`가 `TraceMeta`를 상속하지 않았으므로 오류가 발생한다. 이론적으론 메타클래스 상속을 활용해 `OtherMeta`가 `TraceMeta`를 상속하면 문제를 해결할 수 있다.

In [ ]:
class TraceMeta(type):
    def __new__(meta, name, bases, class_dict):
        klass = super().__new__(meta, name, bases, class_dict)

        for key in dir(klass):
            value = getattr(klass, key)
            if isinstance(value, trace_types):
                wrapped = trace_func(value)
                setattr(klass, key, wrapped)

        return klass
    
class OtherMeta(type):
    pass

class SimpleDict(dict, metaclass=OtherMeta):
    pass

class TraceDict(SimpleDict, metaclass=TraceMeta):
    pass

trace_dict = TraceDict([('안녕', 1)])
trace_dict['거기'] = 2
trace_dict['안녕']
try:
    trace_dict['존재하지 않음']
except KeyError:
    pass # 키 오류가 발생할 것으로 예상함

하지만 라이브러리에 있는 메타클래스를 사용하는 경우엔 코드를 변경할 수 없어서 이 방법은 불가능하다. 또 `TraceMeta` 같은 유틸리티 메타클래스를 여러개 사용하고 싶은 경우에도 사용이 제약된다.

이 문제를 해결하고자 파이썬에서는 [클래스 데코레이터](https://dojang.io/mod/page/view.php?id=2430)를 지원한다. [함수 데코레이터](https://dojang.io/mod/page/view.php?id=2427)처럼 사용 가능하며 클래스 선언 앞에 `@`와 데코레이터 함수를 적으면 된다.

In [ ]:
def my_class_decorator(klass):
    klass.extra_param = '안녕'
    return klass

@my_class_decorator
class MyClass:
    pass

print(MyClass)
print(MyClass.extra_param)

이제 `TraceMeta.__new__` 메소드의 핵심 부분을 별도의 함수로 옮겨서 클래스 데코레이터를 만들면 된다.

In [ ]:
def trace(klass):
    for key in dir(klass):
        value = getattr(klass, key)
        if isinstance(value, trace_types):
            wrapped = trace_func(value)
            setattr(klass, key, wrapped)
    return klass

@trace
class TraceDict(dict):
    pass

trace_dict = TraceDict([('안녕', 1)])
trace_dict['거기'] = 2
trace_dict['안녕']
try:
    trace_dict['존재하지 않음']
except KeyError:
    pass # 키 오류가 발생할 것으로 예상함

이젠 해당 클래스에 이미 메타클래스가 있어도 데코레이터 사용이 가능해진다.

In [ ]:
class OtherMeta(type):
    pass

@trace
class TraceDict(dict, metaclass=TraceMeta):
    pass

trace_dict = TraceDict([('안녕', 1)])
trace_dict['거기'] = 2
trace_dict['안녕']
try:
    trace_dict['존재하지 않음']
except KeyError:
    pass # 키 오류가 발생할 것으로 예상함